в CNN_PATH можете вставить любой csv с предсказаниями рто, как заглушка. оно все равно в итоге не используется

In [ ]:
import optuna
import pandas as pd
import numpy as np
import lightgbm as lgb
import re
import warnings
from scipy.cluster.hierarchy import linkage, fcluster

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

TRAIN_PATH  = '/kaggle/input/datasets/levkremlev/dataset-x5/train_2.csv'
CNN_PATH    = '/kaggle/input/datasets/levkremlev/nn-dataset/test_3_nn.csv'
TARGET_DATE = pd.Timestamp('2025-03-01')
VALID_DATE  = pd.Timestamp('2025-02-01')

def competition_score(mape):
    return 100 * ((100 - min(mape, 100)) / 100) ** 2

def calc_mape(y_true, y_pred):
    mask = y_true > 0
    return 100 * np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask]))

def apply_coef(preds):
    return np.where(
        preds < 70_000_000,
        preds * 1.044,
        preds * 1.038
    ) * 0.992

def save_sub(preds, ids, name, note=''):
    sub = pd.DataFrame({'new_id': ids, 'rto': np.round(preds, 2)})
    sub.to_csv(f'/kaggle/working/{name}.csv', index=False)
    print(f"  ✅ {name}: mean={preds.mean():,.0f}  median={np.median(preds):,.0f}  {note}")

# ════════════════════════════════════════════════════════════
# 1. ЗАГРУЗКА
# ════════════════════════════════════════════════════════════
print("📥 Загрузка...")
df_raw = pd.read_csv(TRAIN_PATH).rename(columns={'РТО': 'target'})
df_raw['timestamp'] = pd.to_datetime(
    df_raw['Год'].astype(str) + '-' +
    df_raw['Месяц'].astype(str).str.zfill(2) + '-01'
)
all_ids = df_raw['new_id'].unique()
print(f"  Магазинов: {len(all_ids)}, строк: {len(df_raw)}")

region_col = None
for c in df_raw.columns:
    if 'егион' in str(c) or 'egion' in str(c).lower():
        region_col = c
        break
if region_col:
    print(f"  Регион: '{region_col}'")

# ════════════════════════════════════════════════════════════
# 2. КЛАСТЕРИЗАЦИЯ
# ════════════════════════════════════════════════════════════
print("\n🔵 Кластеризация...")
df_hist = df_raw[df_raw['timestamp'] < TARGET_DATE].copy()

store_profiles = df_hist.groupby('new_id')['target'].agg(
    mean_rto='mean', std_rto='std', median_rto='median',
    max_rto='max', min_rto='min', count='count',
).fillna(0)
store_profiles['cv']         = store_profiles['std_rto'] / (store_profiles['mean_rto'] + 1)
store_profiles['log_mean']   = np.log1p(store_profiles['mean_rto'])
store_profiles['log_median'] = np.log1p(store_profiles['median_rto'])

zero_counts = df_hist.groupby('new_id')['target'].apply(lambda x: (x == 0).sum())
store_profiles['zero_months'] = store_profiles.index.map(zero_counts).fillna(0)
store_profiles['zero_ratio']  = store_profiles['zero_months'] / store_profiles['count']

last_ts  = sorted(df_hist['timestamp'].unique())
recent_6 = last_ts[-6:]
older_6  = last_ts[-12:-6] if len(last_ts) >= 12 else last_ts[:-6]
mean_recent = df_hist[df_hist['timestamp'].isin(recent_6)].groupby('new_id')['target'].mean()
mean_older  = df_hist[df_hist['timestamp'].isin(older_6)].groupby('new_id')['target'].mean()
store_profiles['trend_6m'] = (
    (mean_recent - mean_older) / (mean_older + 1)
).reindex(store_profiles.index).fillna(0)

march_data = df_hist[df_hist['timestamp'].dt.month == 3].groupby('new_id')['target'].mean()
feb_data   = df_hist[df_hist['timestamp'].dt.month == 2].groupby('new_id')['target'].mean()
store_profiles['mar_vs_feb'] = (
    march_data / (feb_data + 1)
).reindex(store_profiles.index).fillna(1.0)

log_means     = store_profiles['log_mean'].values
q25, q50, q75 = np.percentile(log_means, [25, 50, 75])

def assign_lv1(v):
    if v <= q25:   return 1
    elif v <= q50: return 2
    elif v <= q75: return 3
    else:          return 4

store_profiles['lv1_cluster'] = store_profiles['log_mean'].apply(assign_lv1)

hier_features = ['log_mean', 'cv', 'zero_ratio', 'trend_6m', 'mar_vs_feb']
cluster_final = {}

for lv1 in [1, 2, 3, 4]:
    ids_in = store_profiles[store_profiles['lv1_cluster'] == lv1].index
    n_sub  = 2 if lv1 < 4 else 4
    if len(ids_in) < n_sub * 2:
        for sid in ids_in:
            cluster_final[sid] = f'{lv1}_1'
        continue
    sub_data = store_profiles.loc[ids_in, hier_features].fillna(0).values
    sub_std  = sub_data.std(axis=0)
    sub_std[sub_std == 0] = 1
    sub_norm   = (sub_data - sub_data.mean(axis=0)) / sub_std
    sim_matrix = np.dot(sub_norm, sub_norm.T)
    try:
        Z          = linkage(sim_matrix, method='ward')
        sub_labels = fcluster(Z, t=n_sub, criterion='maxclust')
        for sid, label in zip(ids_in, sub_labels):
            cluster_final[sid] = f'{lv1}_{label}'
    except Exception:
        for sid in ids_in:
            cluster_final[sid] = f'{lv1}_1'

cluster_series  = pd.Series(cluster_final, name='cluster')
cluster_codes   = {c: i for i, c in enumerate(sorted(cluster_series.unique()))}
cluster_numeric = cluster_series.map(cluster_codes)
print(f"  Кластеров: {len(cluster_codes)}")

# ════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════
def build_features(df):
    df  = df.copy().sort_values(['new_id', 'timestamp']).reset_index(drop=True)
    grp = df.groupby('new_id')['target']

    for lag in [1, 2, 3, 6, 12, 13, 14, 24]:
        df[f'lag_{lag}'] = grp.shift(lag)

    for w in [3, 6, 12]:
        df[f'roll_mean_{w}'] = grp.transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean()
        )
        df[f'roll_std_{w}'] = grp.transform(
            lambda x: x.shift(1).rolling(w, min_periods=2).std()
        )

    for span, name in [(3, '3'), (6, '6'), (12, '12')]:
        df[f'ema_{name}'] = grp.transform(
            lambda x, s=span: x.shift(1).ewm(span=s, adjust=False).mean()
        )

    for alpha, aname in [(0.95, '095'), (0.9, '090'), (0.7, '070'), (0.5, '050')]:
        df[f'ewm_a{aname}'] = grp.transform(
            lambda x, a=alpha: x.shift(1).ewm(alpha=a, adjust=False).mean()
        )

    df['weighted_lag'] = (
        0.50 * grp.shift(1).fillna(0) +
        0.30 * grp.shift(2).fillna(0) +
        0.15 * grp.shift(3).fillna(0) +
        0.05 * grp.shift(6).fillna(0)
    )

    df['yoy_growth']         = df['lag_1']  / (df['lag_12'] + 1e-6)
    df['yoy_ratio_13']       = df['lag_1']  / (df['lag_13'] + 1e-6)
    df['yoy_ratio_2y']       = df['lag_12'] / (df['lag_24'] + 1e-6)
    df['yoy_diff']           = df['lag_1']  - df['lag_13']
    df['mar_feb_ratio_prev'] = df['lag_13'] / (df['lag_14'] + 1e-6)

    df['mom_1m']  = df['lag_1'] / (df['lag_2']  + 1e-6) - 1
    df['mom_3m']  = df['lag_1'] / (df['lag_6']  + 1e-6) - 1
    df['mom_12m'] = df['lag_1'] / (df['lag_13'] + 1e-6) - 1

    store_median         = grp.transform('median')
    store_mean           = grp.transform('mean')
    df['lag1_vs_median'] = df['lag_1'] / (store_median + 1e-6)
    df['lag1_vs_mean']   = df['lag_1'] / (store_mean   + 1e-6)

    df['month']     = df['timestamp'].dt.month
    df['quarter']   = df['timestamp'].dt.quarter
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['is_march']  = (df['month'] == 3).astype(int)
    df['is_q1']     = (df['quarter'] == 1).astype(int)
    df['is_jan']    = (df['month'] == 1).astype(int)

    df['cluster']      = df['new_id'].map(cluster_numeric).fillna(0).astype(int)
    df['zero_ratio_s'] = df['new_id'].map(store_profiles['zero_ratio']).fillna(0)
    df['store_cv']     = df['new_id'].map(store_profiles['cv']).fillna(0)
    df['store_trend']  = df['new_id'].map(store_profiles['trend_6m']).fillna(0)
    df['mar_vs_feb_s'] = df['new_id'].map(store_profiles['mar_vs_feb']).fillna(1)
    df['log_mean_rto'] = df['new_id'].map(store_profiles['log_mean']).fillna(0)

    cl_agg = (
        df.groupby(['cluster', 'timestamp'])['target']
        .agg(cluster_median='median', cluster_mean='mean')
        .reset_index()
    )
    cl_agg['timestamp'] = cl_agg['timestamp'] + pd.DateOffset(months=1)
    df = df.merge(
        cl_agg.rename(columns={
            'cluster_median': 'cluster_median_prev',
            'cluster_mean':   'cluster_mean_prev',
        }),
        on=['cluster', 'timestamp'], how='left'
    )
    df['lag1_vs_cluster'] = df['lag_1'] / (df['cluster_median_prev'] + 1e-6)

    if region_col and region_col in df.columns:
        reg_agg = (
            df.groupby([region_col, 'timestamp'])['target']
            .agg(region_median='median', region_mean='mean')
            .reset_index()
        )
        reg_agg['timestamp'] = reg_agg['timestamp'] + pd.DateOffset(months=1)
        df = df.merge(
            reg_agg.rename(columns={
                'region_median': 'region_median_prev',
                'region_mean':   'region_mean_prev',
            }),
            on=[region_col, 'timestamp'], how='left'
        )
        df['lag1_vs_region'] = df['lag_1'] / (df['region_median_prev'] + 1e-6)

    for c in df.columns:
        if df[c].dtype == 'object' and c not in ['timestamp', 'new_id']:
            df[c] = pd.factorize(df[c].fillna('missing'))[0]

    df = df.rename(
        columns={c: re.sub(r'[^\w]', '_', str(c)) for c in df.columns}
    )
    return df


print("\n🔨 Строим признаки...")
march_rows = pd.DataFrame({'new_id': all_ids, 'timestamp': TARGET_DATE})
df_full    = pd.concat([df_raw, march_rows], ignore_index=True)\
               .sort_values(['new_id', 'timestamp'])\
               .reset_index(drop=True)

df_feat  = build_features(df_full)
EXCLUDE  = {'target', 'timestamp', 'new_id', 'Год', 'Месяц', 'год', 'месяц'}
features = [c for c in df_feat.columns if c not in EXCLUDE]
print(f"✅ Признаков: {len(features)}")

# ── NaN ──
train_all   = df_feat[df_feat['target'].notna()].copy()
col_medians = {}
for f in features:
    if f in train_all.columns and pd.api.types.is_numeric_dtype(train_all[f]):
        col_medians[f] = train_all[f].median()

def fill_na(df_in):
    df_out = df_in.copy()
    for f, med in col_medians.items():
        if f in df_out.columns:
            df_out[f] = df_out[f].fillna(med)
    return df_out

train_v = fill_na(df_feat[
    (df_feat['timestamp'] < VALID_DATE) & (df_feat['target'].notna())
])
val_v   = fill_na(df_feat[df_feat['timestamp'] == VALID_DATE])
y_val   = val_v['target'].values

X_train   = train_v[features].values
y_train   = np.log1p(train_v['target'].values)
X_val     = val_v[features].values
y_val_log = np.log1p(val_v['target'].values)

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# ════════════════════════════════════════════════════════════
# 4. OPTUNA
# ════════════════════════════════════════════════════════════
N_TRIALS  = 100
N_ROUNDS  = 2000
ES_ROUNDS = 100

def objective(trial):
    params = {
        'objective':         'regression_l1',
        'metric':            'mape',
        'verbose':           -1,
        'seed':              42,
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'max_depth':         trial.suggest_int('max_depth', 4, 16),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 300),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
        'max_bin':           trial.suggest_int('max_bin', 128, 512),
        'path_smooth':       trial.suggest_float('path_smooth', 0.0, 1.0),
    }

    dtrain = lgb.Dataset(
        X_train, label=y_train,
        params={'max_bin': params['max_bin']},
        free_raw_data=False,
    )
    dval = lgb.Dataset(
        X_val, label=y_val_log,
        params={'max_bin': params['max_bin']},
        reference=dtrain,
        free_raw_data=False,
    )

    model = lgb.train(
        params, dtrain,
        num_boost_round=N_ROUNDS,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(ES_ROUNDS, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )

    preds = np.expm1(model.predict(X_val)).clip(0)
    mape  = calc_mape(y_val, preds)
    trial.set_user_attr('best_iteration', model.best_iteration)
    return mape


print(f"\n🔬 Optuna: {N_TRIALS} trials...")

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=15),
)

study.enqueue_trial({
    'num_leaves':        124,
    'max_depth':         12,
    'min_child_samples': 129,
    'learning_rate':     0.02585488501264429,
    'feature_fraction':  0.531610137661962,
    'bagging_fraction':  0.7475039954480004,
    'bagging_freq':      2,
    'reg_alpha':         0.004177468127523882,
    'reg_lambda':        0.006185664836995357,
    'min_gain_to_split': 0.0,
    'max_bin':           255,
    'path_smooth':       0.0,
})

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    catch=(Exception,),
)

# ════════════════════════════════════════════════════════════
# 5. ИЗВЛЕКАЕМ ЛУЧШИЕ ПАРАМЕТРЫ — до любого падения
# ════════════════════════════════════════════════════════════
trials_ok = [t for t in study.trials if t.value is not None]
trials_ok.sort(key=lambda t: t.value)
best = trials_ok[0]

print(f"\n{'='*60}")
print(f"✅ Лучший trial #{best.number}")
print(f"   MAPE:      {best.value:.4f}%")
print(f"   Балл:      {competition_score(best.value):.2f}")
print(f"   best_iter: {best.user_attrs.get('best_iteration', 'n/a')}")
print(f"{'='*60}")

# Собираем BEST_PARAMS прямо здесь
BEST_PARAMS = {
    'objective':        'regression_l1',
    'metric':           'mape',
    'seed':             42,
    'verbose':          -1,
    **best.params,
}

print("\n📋 BEST_PARAMS:")
for k, v in BEST_PARAMS.items():
    print(f"  '{k}': {v},")

print(f"\n📊 Топ-10 trials:")
for t in trials_ok[:10]:
    print(f"  #{t.number:<4} MAPE={t.value:.4f}%  "
          f"балл={competition_score(t.value):.2f}  "
          f"iter={t.user_attrs.get('best_iteration','?')}")

print(f"\n📈 Прогресс:")
best_so_far = 9999
for t in sorted(study.trials, key=lambda x: x.number):
    if t.value and t.value < best_so_far:
        best_so_far = t.value
        print(f"  Trial #{t.number:<4} → {best_so_far:.4f}%  "
              f"{competition_score(best_so_far):.2f} б.")

# ════════════════════════════════════════════════════════════
# 6. ФИНАЛЬНАЯ МОДЕЛЬ с лучшими параметрами
# ════════════════════════════════════════════════════════════
print("\n🚀 Финальная модель с новыми параметрами...")

train_f  = fill_na(train_all)
march_df = fill_na(df_feat[df_feat['timestamp'] == TARGET_DATE])
ids      = march_df['new_id'].values

best_iter        = best.user_attrs.get('best_iteration', 875)
num_rounds_final = int(best_iter * 1.1)
print(f"  Rounds: {num_rounds_final}")

# Dataset для финала с правильным max_bin
dtrain_f = lgb.Dataset(
    train_f[features].values,
    label=np.log1p(train_f['target'].values),
    params={'max_bin': BEST_PARAMS.get('max_bin', 255)},
    free_raw_data=False,
)

model_f = lgb.train(
    BEST_PARAMS,
    dtrain_f,
    num_boost_round=num_rounds_final,
)

lgb_preds_raw = np.expm1(model_f.predict(march_df[features].values)).clip(0)
lgb_preds_out = apply_coef(lgb_preds_raw)

print(f"  Raw:   mean={lgb_preds_raw.mean():,.0f}  median={np.median(lgb_preds_raw):,.0f}")
print(f"  +коэф: mean={lgb_preds_out.mean():,.0f}  median={np.median(lgb_preds_out):,.0f}")

# Валидационная проверка новых параметров
dtrain_v = lgb.Dataset(
    X_train, label=y_train,
    params={'max_bin': BEST_PARAMS.get('max_bin', 255)},
    free_raw_data=False,
)
model_check = lgb.train(BEST_PARAMS, dtrain_v, num_boost_round=best_iter)
val_check   = np.expm1(model_check.predict(X_val)).clip(0)
mape_check  = calc_mape(y_val, val_check)
print(f"\n📊 Val MAPE новые params: {mape_check:.4f}% | {competition_score(mape_check):.2f} б.")

# ════════════════════════════════════════════════════════════
# 7. БЛЕНДИНГ С CNN
# ════════════════════════════════════════════════════════════
print(f"\n🤝 Загружаем CNN: {CNN_PATH}")
cnn_df       = pd.read_csv(CNN_PATH)
cnn_pred_col = [c for c in cnn_df.columns if c != 'new_id'][0]
print(f"  CNN колонка: '{cnn_pred_col}'  строк: {len(cnn_df)}")

merged = (
    pd.DataFrame({'new_id': ids, 'lgb_pred': lgb_preds_raw})
    .merge(
        cnn_df.rename(columns={cnn_pred_col: 'cnn_pred'})[['new_id', 'cnn_pred']],
        on='new_id', how='left'
    )
)
n_miss = merged['cnn_pred'].isna().sum()
if n_miss > 0:
    print(f"  ⚠️  CNN нет для {n_miss} магазинов → заполняем LGB")
merged['cnn_pred'] = merged['cnn_pred'].fillna(merged['lgb_pred'])

lgb_arr = merged['lgb_pred'].values
cnn_arr = merged['cnn_pred'].values

print(f"  LGB: mean={lgb_arr.mean():,.0f}  median={np.median(lgb_arr):,.0f}")
print(f"  CNN: mean={cnn_arr.mean():,.0f}  median={np.median(cnn_arr):,.0f}")

# ════════════════════════════════════════════════════════════
# 8. СОХРАНЕНИЕ ВСЕХ ВАРИАНТОВ
# ════════════════════════════════════════════════════════════
print("\n📦 Сохраняем:")

# LGB соло
save_sub(lgb_preds_out, ids, 'test', '← LGB соло + коэф')
save_sub(lgb_preds_raw, ids, 'test_no_coef', '← LGB без коэф')

# Все варианты блендинга
for alpha in [0.5, 0.4, 0.3, 0.2, 0.1, 0.0]:
    blended = alpha * lgb_arr + (1 - alpha) * cnn_arr
    out     = apply_coef(blended)
    fname   = f'test_lgb{int(alpha*10)}_cnn{int((1-alpha)*10)}'
    save_sub(out, merged['new_id'].values, fname,
             f'LGB×{alpha} + CNN×{1-alpha:.1f} + коэф')

print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ИТОГ                                                        ║
╠══════════════════════════════════════════════════════════════╣
║  Optuna лучший:   {best.value:.4f}%  {competition_score(best.value):.2f} б. (trial #{best.number})  ║
║  Val check:       {mape_check:.4f}%  {competition_score(mape_check):.2f} б.              ║
║  Final rounds:    {num_rounds_final}                                    ║
╠══════════════════════════════════════════════════════════════╣
║  Сабмитить в порядке:                                        ║
║    1. test_lgb3_cnn7.csv   (LGB×0.3 + CNN×0.7)              ║
║    2. test_lgb2_cnn8.csv   (LGB×0.2 + CNN×0.8)              ║
║    3. test_lgb4_cnn6.csv   (LGB×0.4 + CNN×0.6)              ║
║    4. test_lgb0_cnn10.csv  (CNN соло + коэф)                 ║
║    5. test.csv             (LGB соло + коэф)                 ║
╚══════════════════════════════════════════════════════════════╝
""")

отсюда после запуска я взял test_no_coef для чистого lgb

теперь обучаем нейросеть MLP

In [ ]:
#3
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPRegressor
import os, re, warnings
warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════
# 1. КОНСТАНТЫ И ФУНКЦИЯ БАЛЛОВ
# ════════════════════════════════════════════════════════════
TRAIN_PATH = '/kaggle/input/datasets/dexter2007/train-2/train_2.csv'
VALID_DATE = pd.Timestamp('2025-02-01')
TARGET_DATE = pd.Timestamp('2025-03-01')

def get_score(mape):
    return 100 * ((100 - min(mape, 100)) / 100) ** 2

# ════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING (ДЛЯ НЕЙРОСЕТИ - ТОЛЬКО ГЛАВНОЕ)
# ════════════════════════════════════════════════════════════
def build_nn_features(df):
    df = df.copy().sort_values(['new_id', 'timestamp'])
    grp = df.groupby('new_id')['target']
    
    # Нейросети любят сжатые агрегаты
    df['lag_1'] = grp.shift(1)
    df['lag_12'] = grp.shift(12)
    df['roll_mean_3'] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    
    df['month'] = df['timestamp'].dt.month
    df['is_march'] = (df['month'] == 3).astype(int)
    
    # Кодируем категории в числа (Simple Embeddings)
    for col in ['Регион', 'Населенный пункт']:
        if col in df.columns:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))
            
    return df.fillna(0)

# ════════════════════════════════════════════════════════════
# 3. ШАГ 1: ВАЛИДАЦИЯ (Deep Learning Accuracy)
# ════════════════════════════════════════════════════════════
print("📥 Подготовка данных для нейросети...")
df_raw = pd.read_csv(TRAIN_PATH).rename(columns={'РТО': 'target'})
df_raw['timestamp'] = pd.to_datetime(df_raw['Год'].astype(str) + '-' + df_raw['Месяц'].astype(str).str.zfill(2) + '-01')

df_feat = build_nn_features(df_raw)
features = ['lag_1', 'lag_12', 'roll_mean_3', 'month', 'is_march']

# Нормализация (Критически важно для NN!)
scaler_x = StandardScaler()
scaler_y = StandardScaler()

train_idx = (df_feat['timestamp'] < VALID_DATE) & (df_feat['lag_12'] > 0)
val_idx = (df_feat['timestamp'] == VALID_DATE)

X_tr = scaler_x.fit_transform(df_feat.loc[train_idx, features])
y_tr = scaler_y.fit_transform(np.log1p(df_feat.loc[train_idx, 'target']).values.reshape(-1, 1))

X_vl = scaler_x.transform(df_feat.loc[val_idx, features])
y_vl_actual = df_feat.loc[val_idx, 'target'].values

print("🚀 Обучение MLP-нейросети (3 слоя)...")
nn = MLPRegressor(hidden_layer_sizes=(64, 32, 16), activation='relu', solver='adam', 
                  max_iter=200, random_state=42, batch_size=128)
nn.fit(X_tr, y_tr.ravel())

# Прогноз и обратное масштабирование (без коэффициента 0.992)
preds_nn_scaled = nn.predict(X_vl)
preds_nn_log = scaler_y.inverse_transform(preds_nn_scaled.reshape(-1, 1))
preds_nn = np.expm1(preds_nn_log).ravel()

mape_nn = np.mean(np.abs((y_vl_actual - preds_nn) / y_vl_actual)) * 100
print(f"📊 NN MODEL (Val): MAPE = {mape_nn:.4f}% | Score = {get_score(mape_nn):.2f}")

# ════════════════════════════════════════════════════════════
# 4. ФИНАЛЬНЫЙ ПРОГНОЗ (МАРТ 2025)
# ════════════════════════════════════════════════════════════
print("\n🚀 Генерация финального прогноза нейросетью...")
march_grid = pd.DataFrame({'new_id': df_raw['new_id'].unique(), 'timestamp': TARGET_DATE})
df_full = build_nn_features(pd.concat([df_raw, march_grid], ignore_index=True))

# Обучаем на всём
X_all = scaler_x.fit_transform(df_full[df_full['target'] > 0][features])
y_all = scaler_y.fit_transform(np.log1p(df_full[df_full['target'] > 0]['target']).values.reshape(-1, 1))
X_march = scaler_x.transform(df_full[df_full['timestamp'] == TARGET_DATE][features])

nn_final = MLPRegressor(hidden_layer_sizes=(64, 32, 16), max_iter=250, random_state=42)
nn_final.fit(X_all, y_all.ravel())

p_march_scaled = nn_final.predict(X_march)
p_march_log = scaler_y.inverse_transform(p_march_scaled.reshape(-1, 1))
p_march = np.expm1(p_march_log).ravel() # без коэффициента 0.990

In [ ]:
sub = pd.DataFrame({'new_id': march_grid['new_id'], 'rto': p_march.round(2)})
sub.to_csv('test_3_nn.csv', index=None)
print("✅ Файл test_3_nn.csv создан!")

берется test_3_nn.csv 

теперь обучение ridge-regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
import optuna

# Отключаем логирование Optuna, чтобы вывод оставался чистым
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

TRAIN_PATH  = '/kaggle/input/datasets/levkremlev/dataset-x5/train_2.csv'
TARGET_DATE = pd.Timestamp('2025-03-01')

print("📥 Загрузка...")
df_raw = pd.read_csv(TRAIN_PATH).rename(columns={'РТО': 'target'})
df_raw['timestamp'] = pd.to_datetime(
    df_raw['Год'].astype(str) + '-' +
    df_raw['Месяц'].astype(str).str.zfill(2) + '-01'
)
all_ids = df_raw['new_id'].unique()
print(f"  Магазинов: {len(all_ids)}, строк: {len(df_raw)}")

# ════════════════════════════════════════════════════════════
# FEATURE ENGINEERING — только лаги и EMA
# ════════════════════════════════════════════════════════════
print("\n🔨 Строим признаки...")

march_rows = pd.DataFrame({'new_id': all_ids, 'timestamp': TARGET_DATE})
df_full    = pd.concat([df_raw, march_rows], ignore_index=True)\
               .sort_values(['new_id', 'timestamp'])\
               .reset_index(drop=True)

grp = df_full.groupby('new_id')['target']

# Лаги
for lag in [1, 2, 3, 6, 12, 13]:
    df_full[f'lag_{lag}'] = grp.shift(lag)

# Rolling
for w in [3, 6, 12]:
    df_full[f'roll_mean_{w}'] = grp.transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).mean()
    )

# EMA
for span, name in [(3, '3'), (6, '6'), (12, '12')]:
    df_full[f'ema_{name}'] = grp.transform(
        lambda x, s=span: x.shift(1).ewm(span=s, adjust=False).mean()
    )

# Сезонность
df_full['month']     = df_full['timestamp'].dt.month
df_full['month_sin'] = np.sin(2 * np.pi * df_full['month'] / 12)
df_full['month_cos'] = np.cos(2 * np.pi * df_full['month'] / 12)

FEATURES = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13',
    'roll_mean_3', 'roll_mean_6', 'roll_mean_12',
    'ema_3', 'ema_6', 'ema_12',
    'month', 'month_sin', 'month_cos',
]

print(f"✅ Признаков: {len(FEATURES)}")

# ════════════════════════════════════════════════════════════
# ПОДГОТОВКА ДАННЫХ
# ════════════════════════════════════════════════════════════
train_df = df_full[df_full['target'].notna()].copy()
march_df = df_full[df_full['timestamp'] == TARGET_DATE].copy()

# Заполняем NaN медианой по признаку
for f in FEATURES:
    med = train_df[f].median()
    train_df[f] = train_df[f].fillna(med)
    march_df[f] = march_df[f].fillna(med)

X_train = train_df[FEATURES].values
y_train = np.log1p(train_df['target'].values)
X_march = march_df[FEATURES].values

# Масштабируем
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_march = scaler.transform(X_march)

# ════════════════════════════════════════════════════════════
# НАСТРОЙКА OPTUNA (Оптимизация под RMSE на логарифмах)
# ════════════════════════════════════════════════════════════
print("\n🔍 Подбор параметров Ridge через Optuna (оптимизация RMSE на логах)...")

# Выделяем последний месяц из обучающей выборки для локальной валидации
last_train_date = train_df['timestamp'].max()
val_mask = (train_df['timestamp'] == last_train_date).values
train_mask = (train_df['timestamp'] < last_train_date).values

X_tr, y_tr = X_train[train_mask], y_train[train_mask]
X_val, y_val = X_train[val_mask], y_train[val_mask]

# Целевая функция для подбора параметров
def objective(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 1e4, log=True)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    solver = trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'])
    
    model = Ridge(alpha=alpha, fit_intercept=fit_intercept, solver=solver, random_state=42)
    model.fit(X_tr, y_tr)
    
    # Считаем RMSE прямо на логарифмированных прогнозах
    preds_log = model.predict(X_val)
    rmse = np.sqrt(np.mean((y_val - preds_log) ** 2))
    return rmse

# Запускаем поиск (минимизируем RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"  Лучшие параметры: {best_params}")
print(f"  Лучший RMSE на локальной валидации: {study.best_value:.5f}")

# ════════════════════════════════════════════════════════════
# ИТОГОВОЕ ОБУЧЕНИЕ И ПРЕДСКАЗАНИЕ
# ════════════════════════════════════════════════════════════
print("\n🔵 Обучаем итоговый Ridge на всех данных с лучшими параметрами...")
print(f"  Train: {len(X_train)} строк")

# Обучаем модель с найденными параметрами на 100% тренировочных данных
best_ridge = Ridge(**best_params, random_state=42)
best_ridge.fit(X_train, y_train)

# Прогноз на март 2025
ridge_preds = np.expm1(best_ridge.predict(X_march)).clip(0)

print(f"  mean={ridge_preds.mean():,.0f}  median={np.median(ridge_preds):,.0f}")

# ════════════════════════════════════════════════════════════
# СОХРАНЕНИЕ
# ════════════════════════════════════════════════════════════
ridge_sub = pd.DataFrame({
    'new_id': march_df['new_id'].values,
    'rto':    np.round(ridge_preds, 2)
})
ridge_sub.to_csv('/kaggle/working/test_ridge.csv', index=False)
print(f"\n✅ Сохранено: test_ridge.csv  ({len(ridge_sub)} строк)")

отсюда нужно будет взять файл test_ridge.csv

итоговый блендинг(используем файлы от lgbm, mlp и ridge regression):

In [ ]:
import pandas as pd
import numpy as np

def apply_coef(preds):
    return np.where(
        preds < 70_000_000,
        preds * 1.044,
        preds * 1.038
    ) * 0.992

def save_sub(preds, ids, name, note=''):
    pd.DataFrame({'new_id': ids, 'rto': np.round(preds, 2)}).to_csv(
        f'/kaggle/working/{name}.csv', index=False
    )
    print(f"  ✅ {name}: mean={preds.mean():,.0f}  median={np.median(preds):,.0f}  {note}")

lgb_df    = pd.read_csv('/kaggle/input/datasets/levkremlev/lgbm-no-coefs/test_no_coef.csv')
mlp_df    = pd.read_csv('/kaggle/input/datasets/levkremlev/mlp-no-coefs/MLPNOCOEF.csv')
ridge_df  = pd.read_csv('/kaggle/input/datasets/levkremlev/ridge-no-coef/test_ridge.csv')

lgb_col   = [c for c in lgb_df.columns   if c != 'new_id'][0]
mlp_col   = [c for c in mlp_df.columns   if c != 'new_id'][0]
ridge_col = [c for c in ridge_df.columns if c != 'new_id'][0]

merged = (
    lgb_df.rename(columns={lgb_col: 'lgb'})
    .merge(mlp_df.rename(columns={mlp_col: 'mlp'})[['new_id','mlp']],
           on='new_id', how='left')
    .merge(ridge_df.rename(columns={ridge_col: 'ridge'})[['new_id','ridge']],
           on='new_id', how='left')
)

lgb_arr   = merged['lgb'].values
mlp_arr   = merged['mlp'].fillna(merged['lgb']).values
ridge_arr = merged['ridge'].fillna(merged['lgb']).values
ids       = merged['new_id'].values

print("📊 Статистика:")
print(f"  LGB:   mean={lgb_arr.mean():,.0f}")
print(f"  MLP:   mean={mlp_arr.mean():,.0f}")
print(f"  Ridge: mean={ridge_arr.mean():,.0f}")
print(f"  Рекорд ориентир: mean=103,416,696")

print("\n📦 Варианты (Ridge максимум 5%):")

combos = [
    # (lgb_w, mlp_w, ridge_w, name)
    # Вокруг рекордного lgb43_mlp52_r05 (mean=103,957)
    (0.44, 0.53, 0.03, 'lgb44_mlp53_r03'),
    (0.43, 0.54, 0.03, 'lgb43_mlp54_r03'),
    (0.44, 0.51, 0.05, 'lgb44_mlp51_r05'),
    (0.42, 0.53, 0.05, 'lgb42_mlp53_r05'),
    (0.40, 0.55, 0.05, 'lgb40_mlp55_r05'),
]

for lgb_w, mlp_w, ridge_w, name in combos:
    blend = lgb_w * lgb_arr + mlp_w * mlp_arr + ridge_w * ridge_arr
    out   = apply_coef(blend)
    save_sub(out, ids, f'test_{name}',
             f'lgb×{lgb_w} mlp×{mlp_w} ridge×{ridge_w}')

теперь берем файл lgb44_mlp51_r05.csv - именно он выдает наш лучший скор.